# Restaurant Branch Performance — Exploratory Data Analysis (EDA)

**Day 13 Assignment — Pandas**

This notebook performs a complete exploratory data analysis of the Restaurant Branch Performance Dataset.  
The analysis covers:

- Data loading, structure, data types, missing values, and duplicates
- Summary statistics and descriptive analysis
- Distribution analysis of important numerical variables
- Branch, region, and store-type comparisons
- Correlation analysis using a numerical correlation matrix and heatmap
- Data-driven observations and conclusions

> **Note:** The analysis focuses on customers, revenue, profit, marketing spend, staff count, delivery time, and customer ratings, as required by the assignment.


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Libraries imported successfully.")


## 1. Load the Dataset

The cell below works in both Google Colab and a local Jupyter environment.  
In Colab, upload the CSV when prompted. If the supplied filename already exists, it will be used automatically.


In [ ]:
# Locate the dataset
import os

possible_paths = [
    "Day13_Restaurant_Branch_Performance_Dataset(1).csv",
    "Day13_Restaurant_Branch_Performance_Dataset.csv",
    "/mnt/data/Day13_Restaurant_Branch_Performance_Dataset(1).csv"
]

csv_path = next((p for p in possible_paths if os.path.exists(p)), None)

if csv_path is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_path = next(iter(uploaded))
    except Exception:
        raise FileNotFoundError(
            "CSV not found. Please upload the Restaurant Branch Performance Dataset."
        )

df = pd.read_csv(csv_path)
print(f"Dataset loaded from: {csv_path}")
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]:,}")


## 2. Initial Data Inspection

In [ ]:
# First five rows
df.head()


In [ ]:
# Last five rows
df.tail()


In [ ]:
# Dataset shape
print("Shape:", df.shape)

# Column names
print("\nColumns:")
print(df.columns.tolist())

# Data types and non-null counts
print("\nData types and non-null counts:")
df.info()


## 3. Data Quality Check

In [ ]:
# Missing values
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)

missing_table = pd.DataFrame({
    "Missing_Count": missing,
    "Missing_Percentage": missing_pct
})

missing_table[missing_table["Missing_Count"] > 0]


In [ ]:
# Duplicate rows
print("Number of fully duplicated rows:", df.duplicated().sum())

# Unique values in categorical columns
categorical_cols = df.select_dtypes(include="object").columns
unique_counts = df[categorical_cols].nunique().sort_values(ascending=False)
unique_counts


## 4. Descriptive / Summary Statistics

`describe()` is used to understand central tendency, spread, and range. Both numerical and categorical summaries are included.


In [ ]:
# Numerical descriptive statistics
df.describe().T


In [ ]:
# Categorical descriptive statistics
df.describe(include="object").T


## 5. Important Business Metrics

The following summary focuses on the variables highlighted in the assignment.


In [ ]:
key_metrics = [
    "Customers", "Revenue", "Profit", "Marketing_Spend",
    "Staff_Count", "Avg_Delivery_Min", "Customer_Rating"
]

df[key_metrics].describe().T


## 6. Distribution Analysis

In [ ]:
# Histograms for important numerical variables
key_metrics = [
    "Customers", "Revenue", "Profit", "Marketing_Spend",
    "Staff_Count", "Avg_Delivery_Min", "Customer_Rating"
]

fig, axes = plt.subplots(4, 2, figsize=(14, 18))
axes = axes.flatten()

for i, col in enumerate(key_metrics):
    sns.histplot(df[col], kde=True, ax=axes[i])
    axes[i].set_title(f"Distribution of {col}")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Frequency")

axes[-1].axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# Boxplots to inspect spread and possible outliers
fig, axes = plt.subplots(4, 2, figsize=(14, 18))
axes = axes.flatten()

for i, col in enumerate(key_metrics):
    sns.boxplot(x=df[col], ax=axes[i])
    axes[i].set_title(f"Boxplot of {col}")
    axes[i].set_xlabel(col)

axes[-1].axis("off")
plt.tight_layout()
plt.show()


## 7. Categorical Analysis

In [ ]:
# Frequency counts for key categorical variables
for col in ["Branch", "Region", "Store_Type", "Weather", "Promotion"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))


In [ ]:
# Number of records by Branch
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x="Branch", order=df["Branch"].value_counts().index)
plt.title("Number of Records by Branch")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 8. Performance Comparison by Branch

In [ ]:
branch_summary = (
    df.groupby("Branch")[
        ["Customers", "Revenue", "Profit", "Marketing_Spend",
         "Staff_Count", "Avg_Delivery_Min", "Customer_Rating"]
    ]
    .mean()
    .sort_values("Profit", ascending=False)
    .round(2)
)

branch_summary


In [ ]:
# Average profit by branch
plt.figure(figsize=(11, 6))
sns.barplot(
    data=branch_summary.reset_index(),
    x="Branch",
    y="Profit",
    order=branch_summary.index
)
plt.title("Average Profit by Branch")
plt.xlabel("Branch")
plt.ylabel("Average Profit")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Average revenue by branch
plt.figure(figsize=(11, 6))
sns.barplot(
    data=branch_summary.reset_index(),
    x="Branch",
    y="Revenue",
    order=branch_summary.sort_values("Revenue", ascending=False).index
)
plt.title("Average Revenue by Branch")
plt.xlabel("Branch")
plt.ylabel("Average Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 9. Performance Comparison by Region

In [ ]:
region_summary = (
    df.groupby("Region")[
        ["Customers", "Revenue", "Profit", "Marketing_Spend",
         "Staff_Count", "Avg_Delivery_Min", "Customer_Rating"]
    ]
    .mean()
    .sort_values("Profit", ascending=False)
    .round(2)
)

region_summary


In [ ]:
# Revenue and profit by region
region_plot = df.groupby("Region")[["Revenue", "Profit"]].mean().reset_index()
region_plot_melted = region_plot.melt(
    id_vars="Region", var_name="Metric", value_name="Average_Value"
)

plt.figure(figsize=(10, 6))
sns.barplot(data=region_plot_melted, x="Region", y="Average_Value", hue="Metric")
plt.title("Average Revenue and Profit by Region")
plt.tight_layout()
plt.show()


## 10. Performance Comparison by Store Type

In [ ]:
store_type_summary = (
    df.groupby("Store_Type")[
        ["Customers", "Revenue", "Profit", "Marketing_Spend",
         "Staff_Count", "Avg_Delivery_Min", "Customer_Rating"]
    ]
    .mean()
    .sort_values("Profit", ascending=False)
    .round(2)
)

store_type_summary


In [ ]:
# Average revenue and profit by store type
store_plot = df.groupby("Store_Type")[["Revenue", "Profit"]].mean().reset_index()
store_plot_melted = store_plot.melt(
    id_vars="Store_Type", var_name="Metric", value_name="Average_Value"
)

plt.figure(figsize=(10, 6))
sns.barplot(data=store_plot_melted, x="Store_Type", y="Average_Value", hue="Metric")
plt.title("Average Revenue and Profit by Store Type")
plt.tight_layout()
plt.show()


## 11. Relationship Analysis

Scatter plots help inspect relationships between operational/business variables.  
The trend line is descriptive only and **does not imply causation**.


In [ ]:
# Customers vs Revenue
plt.figure(figsize=(8, 6))
sns.regplot(data=df, x="Customers", y="Revenue", scatter_kws={"alpha": 0.6})
plt.title("Customers vs Revenue")
plt.tight_layout()
plt.show()


In [ ]:
# Marketing Spend vs Revenue
plt.figure(figsize=(8, 6))
sns.regplot(data=df, x="Marketing_Spend", y="Revenue", scatter_kws={"alpha": 0.6})
plt.title("Marketing Spend vs Revenue")
plt.tight_layout()
plt.show()


In [ ]:
# Delivery Time vs Customer Rating
plt.figure(figsize=(8, 6))
sns.regplot(data=df, x="Avg_Delivery_Min", y="Customer_Rating", scatter_kws={"alpha": 0.6})
plt.title("Average Delivery Time vs Customer Rating")
plt.tight_layout()
plt.show()


## 12. Correlation Analysis

A Pearson correlation matrix is calculated for numerical variables.  
Values close to **+1** indicate strong positive linear association, values close to **-1** indicate strong negative linear association, and values near **0** indicate weak linear association.


In [ ]:
# Numerical correlation matrix
numeric_cols = df.select_dtypes(include=np.number).columns
corr_matrix = df[numeric_cols].corr()

corr_matrix.round(3)


In [ ]:
# Correlation heatmap
plt.figure(figsize=(15, 11))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5
)
plt.title("Correlation Matrix of Numerical Variables")
plt.tight_layout()
plt.show()


In [ ]:
# Strongest unique correlations (excluding self-correlations)
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
strongest_pairs = (
    upper.stack()
    .sort_values(key=lambda s: s.abs(), ascending=False)
    .to_frame("Pearson_Correlation")
)

strongest_pairs.head(15).round(3)


## 13. Additional Business Indicators

In [ ]:
# Profit margin for each observation
df["Profit_Margin"] = (df["Profit"] / df["Revenue"]) * 100

print("Average profit margin:", df["Profit_Margin"].mean().round(2), "%")
print("Overall profit margin:", (df["Profit"].sum() / df["Revenue"].sum() * 100).round(2), "%")

# Average profit margin by store type
df.groupby("Store_Type")["Profit_Margin"].mean().sort_values(ascending=False).round(2)


In [ ]:
# Customer-to-order conversion proxy
df["Order_Rate_Per_Customer"] = (df["Orders"] / df["Customers"]) * 100

print("Average orders per customer:", (df["Orders"] / df["Customers"]).mean().round(3))
df.groupby("Store_Type")["Order_Rate_Per_Customer"].mean().sort_values(ascending=False).round(2)


## 14. Key Data-Driven Observations

The following observations are based directly on the calculated descriptive statistics, group comparisons, and Pearson correlation matrix.


1. Customer traffic is strongly associated with order volume: Customers and Orders have a Pearson correlation of 0.980, indicating that branches serving more customers generally process more orders.
2. Revenue has a very strong positive relationship with Profit (r = 0.967), so higher-revenue observations generally correspond to higher absolute profit.
3. Average revenue is highest for the Premium store type (₹152,867 per observation), while Express stores average only ₹52,327. This shows a substantial performance gap by store format.
4. Among branches, Bengaluru has the highest average profit (₹31,891), while Mumbai has the lowest (₹26,581), showing meaningful branch-level variation.
5. Marketing spend has a moderate positive correlation with Customers (r = 0.405) and Revenue (r = 0.431), suggesting that higher-spend observations tend to have higher traffic and sales, although correlation does not establish causation.
6. Customer ratings are tightly concentrated around a high level (mean 4.47, median 4.47), and their correlation with Revenue is 0.008; ratings therefore show little linear relationship with sales in this dataset.
7. Average delivery time is also fairly stable (mean 26.10 minutes), with a weak correlation of -0.437 with ratings, suggesting delivery speed is not a major linear driver of ratings here.
8. Only the Promotion column contains missing values (184 of 350 rows), while the dataset contains 0 fully duplicated rows. The missing promotion values should be treated as 'no recorded promotion' only if that meaning is confirmed by the data documentation.

## 15. Conclusion

The EDA shows that restaurant performance varies meaningfully by branch and especially by store type. Customer volume, orders, revenue, and profit move together strongly, while marketing spend has a moderate positive association with customer traffic and revenue. Premium stores stand out with substantially higher average customers, revenue, and profit than Standard and Express formats. At the same time, customer ratings and delivery times are comparatively stable and show weak linear relationships with the major financial measures.

Overall, the analysis suggests that **sales volume and store format are important indicators of absolute financial performance**, while operational service measures such as delivery time and customer rating are less strongly connected to revenue/profit in this particular dataset. Correlation should be interpreted as association rather than proof of cause-and-effect.


---
### Submission Note
This notebook contains the complete EDA workflow using Pandas, Matplotlib, and Seaborn. Run all cells from top to bottom before submission so that the tables, charts, correlation matrix, and observations are visible in the notebook.
